In [24]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# 경로 설정
base_dir = Path(r"C:\ai_x\source\Pks_Develop\햄버거코드")
input_dir = base_dir / "햄버거샌드위치디저트메뉴이미지월별모음"
output_dir = base_dir / "햄버거샌드위치디저트메뉴이미지월별모음_비중반영"
comp_path = base_dir / "상세메뉴_보정결과_비중포함_출력.xlsx"

# 구성비 기준 파일 불러오기
df_comp = pd.read_excel(comp_path)

# 작업할 엑셀 파일 목록
excel_files = sorted(input_dir.glob("menu_images_*.xlsx"))

# 전체 처리 루프
for file_path in excel_files:
    print(f"[처리중] {file_path.name}")
    
    # 연월 추출
    match = re.search(r'(\d{6})', file_path.name)
    if not match:
        print(f"연월 추출 실패: {file_path.name}")
        continue
    ym = match.group(1)
    month_key = f"{ym[:4]}-{ym[4:]}"
    weight_col = f"{month_key}_비중"

    if weight_col not in df_comp.columns:
        print(f"{weight_col} 없음 - 건너뜀")
        continue

    # 구성비 딕셔너리
    comp_dict = df_comp.set_index("상세메뉴")[weight_col].dropna().to_dict()

    # 원본 엑셀 파일 열기
    xls = pd.ExcelFile(file_path)
    if "URL목록" not in xls.sheet_names or "요약" not in xls.sheet_names:
        print(f"필수 시트 누락 - {file_path.name}")
        continue

    df_url = xls.parse("URL목록")
    df_summary = xls.parse("요약")

    if "메뉴" not in df_url.columns or "메뉴" not in df_summary.columns:
        print(f"'메뉴' 열 누락 - {file_path.name}")
        continue

    # 메뉴 그룹화
    menu_group = df_url.groupby("메뉴")

    # 메뉴별 URL 수 기준 최대 메뉴 추출
    menu_counts = df_url["메뉴"].value_counts()
    top_menu = menu_counts.idxmax()
    top_count = menu_counts.max()

    # 적용비중 계산
    adjusted_weights = {}
    for menu in df_url["메뉴"].unique():
        count = menu_counts.get(menu, 0)
        ratio = (count / top_count) * 100 if top_count > 0 else 0
        adjusted = max(10.0, ratio)
        adjusted_weights[menu] = round(adjusted, 2)

    # 비중 기반 할당 수 계산
    allocations = {}
    for menu, group in menu_group:
        base_count = len(group)
        percent = adjusted_weights.get(menu, 10)
        alloc_count = max(int(np.ceil(base_count * percent / 100)), 1)
        allocations[menu] = alloc_count

    # 순서 유지한 샘플링
    ordered_menus = df_url["메뉴"].drop_duplicates().tolist()
    sampled_rows = []
    for menu in ordered_menus:
        if menu in menu_group.groups:
            group = menu_group.get_group(menu)
            count = allocations.get(menu, 0)
            sampled_rows.append(group.iloc[:count])
    df_sampled = pd.concat(sampled_rows)

    # 비교 통계 생성
    original_counts = df_url["메뉴"].value_counts().rename("원본_URL수")
    sampled_counts = df_sampled["메뉴"].value_counts().rename("비중반영_URL수")
    df_compare = pd.concat([original_counts, sampled_counts], axis=1).fillna(0).astype(int)
    df_compare["구성비(%)"] = (df_compare["비중반영_URL수"] / df_compare["원본_URL수"] * 100).round(1)
    df_compare["적용_비중"] = pd.Series(adjusted_weights).reindex(df_compare.index).fillna(10.0)

    # 요약 시트 병합
    df_summary_final = df_summary.merge(
        df_compare.reset_index().rename(columns={"index": "메뉴"}),
        on="메뉴", how="left"
    )

    # 결과 저장
    output_path = output_dir / file_path.name
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_sampled.to_excel(writer, sheet_name="URL목록", index=False)
        df_summary_final.to_excel(writer, sheet_name="요약", index=False)

    print(f"[저장 완료] {output_path.name}")

[처리중] menu_images_202306_20250717_1418.xlsx
[저장 완료] menu_images_202306_20250717_1418.xlsx
[처리중] menu_images_202307_20250717_1958.xlsx
[저장 완료] menu_images_202307_20250717_1958.xlsx
[처리중] menu_images_202308_20250718_0127.xlsx
[저장 완료] menu_images_202308_20250718_0127.xlsx
[처리중] menu_images_202309_20250717_1415.xlsx
[저장 완료] menu_images_202309_20250717_1415.xlsx
[처리중] menu_images_202310_20250717_1955.xlsx
[저장 완료] menu_images_202310_20250717_1955.xlsx
[처리중] menu_images_202311_20250718_0125.xlsx
[저장 완료] menu_images_202311_20250718_0125.xlsx
[처리중] menu_images_202312_20250717_1419.xlsx
[저장 완료] menu_images_202312_20250717_1419.xlsx
[처리중] menu_images_202401_20250717_2003.xlsx
[저장 완료] menu_images_202401_20250717_2003.xlsx
[처리중] menu_images_202402_20250718_0134.xlsx
[저장 완료] menu_images_202402_20250718_0134.xlsx
[처리중] menu_images_202403_20250717_1417.xlsx
[저장 완료] menu_images_202403_20250717_1417.xlsx
[처리중] menu_images_202404_20250717_2000.xlsx
[저장 완료] menu_images_202404_20250717_2000.xlsx
[처리중] menu